In [7]:
import pandas as pd
import os
import re
from pathlib import Path
from collections import defaultdict

class ConceptStrengthTracker:
    def __init__(self, base_path):
        """
        base_path: Path to models directory containing BERT, BOWMAN, LLAMA subdirectories
        Each model should have: model_name/lottery_ticket/X%prune/clusterY.csv
        """
        self.base_path = Path(base_path)
        self.models = ['BERT', 'BOWMAN', 'LLAMA']
        self.model_data = {}
    
    def get_indiv_concepts(self, formula) -> list:
        concepts=[]
        concps = re.findall(r'\b(?:NOT )?(?:pre:tok:|pre:tag:|hyp:tag:|hyp:tok:|oth:)\S*', formula)
        for c in concps:
            try:
                end_idx =c.index(')')
            except:
                end_idx = len(c)
            concepts.append(c[:end_idx])
        return concepts
    def concept_frequeny(self,col):
        freq = defaultdict(int)
        for row in col:
            concepts = self.get_indiv_concepts(row)
            for c in concepts:
                freq[c] += 1
        return freq
    def load_model_data(self, model_name):
        """Load all CSV files for a specific model"""
        print(f"\nLoading {model_name} data...")
        
        
        prune_data = {}
        
        for prune_dir in sorted(self.base_path.iterdir()):
      
            if not prune_dir.is_dir():
                continue
                
            try:
                match = str(prune_dir.name).split("%")
            except:
                continue
                
            prune_level = match[0]
            prune_data[prune_level] = {}
            for csv_file in prune_dir.glob('Cluster*'):
                cluster_match = re.search(r'Cluster(\d+)', csv_file.name)
                if not cluster_match:
                    continue
                    
                cluster = cluster_match.group(1)
                
                try:
                    df = pd.read_csv(csv_file)
                    
                    # Check for required columns
                    if 'unit' not in df.columns or 'best_name' not in df.columns:
                        print(f"  Warning: {csv_file} missing 'unit' or 'formula' columns")
                        continue
                    
                    # Clean data
                    df = df.dropna(subset=['unit', 'best_name'])
                    
                    # Count concepts (concept strength = number of units encoding it)
                    concept_counts = self.concept_frequeny(df['best_name'])
                    
                    prune_data[prune_level][cluster] = {
                        'concept_counts': concept_counts,
                        'total_units': len(df)
                    }
                    
                    print(f"  Loaded: {prune_level}% prune, Cluster {cluster} - {len(concept_counts)} unique concepts, {len(df)} units")
                    
                except Exception as e:
                    print(f"  Error loading {csv_file}: {e}")
        
        print(f"  Total pruning levels loaded: {len(prune_data)}")
        return prune_data
    
    def build_concept_strength_matrix(self, model_name, prune_data):
        """Build concept strength matrices for each cluster"""
        
        print(f"\nBuilding concept strength matrices for {model_name}...")
        
        # Get all unique concepts across all pruning levels for each cluster
        cluster_all_concepts = defaultdict(set)
        
        for prune_level, clusters in prune_data.items():
            for cluster, data in clusters.items():
                for concept in data['concept_counts'].keys():
                    cluster_all_concepts[cluster].add(concept)
        
        # Get sorted prune levels
        prune_levels = sorted(prune_data.keys(), key=lambda x: float(x))
        
        # Build matrix for each cluster
        cluster_matrices = {}
        
        for cluster in sorted(cluster_all_concepts.keys(), key=int):
            concepts = sorted(cluster_all_concepts[cluster])
            
            print(f"  Cluster {cluster}: {len(concepts)} unique concepts across all pruning levels")
            
            # Initialize matrix: rows = concepts, columns = prune levels
            matrix_data = []
            
            for concept in concepts:
                row = {'concept': concept}
                
                for prune_level in prune_levels:
                    if cluster in prune_data[prune_level]:
                        # Get count for this concept at this pruning level
                        count = prune_data[prune_level][cluster]['concept_counts'].get(concept, 0)
                        row[f'{prune_level}%'] = count
                    else:
                        row[f'{prune_level}%'] = 0
                
                matrix_data.append(row)
            
            # Create DataFrame
            df = pd.DataFrame(matrix_data)
            
            # Reorder columns: concept first, then prune levels in order
            columns = ['concept'] + [f'{level}%' for level in prune_levels]
            df = df[columns]
            
            cluster_matrices[cluster] = df
        
        return cluster_matrices
    
    def save_cluster_matrices(self, model_name, cluster_matrices, output_dir='concept_strength_output'):
        """Save concept strength matrices to CSV files"""
        
        output_path = Path(output_dir) / model_name
        output_path.mkdir(parents=True, exist_ok=True)
        
        print(f"\nSaving {model_name} concept strength matrices to {output_path}...")
        
        for cluster, df in cluster_matrices.items():
            filename = f"Cluster{cluster}_conceptstrength.csv"
            filepath = output_path / filename
            
            df.to_csv(filepath, index=False)
            print(f"  Saved: {filename} ({len(df)} concepts)")
        
        return output_path
    
    def generate_summary_statistics(self, model_name, cluster_matrices):
        """Generate summary statistics for concept strength trends"""
        
        print(f"\n{'='*80}")
        print(f"CONCEPT STRENGTH SUMMARY: {model_name}")
        print(f"{'='*80}\n")
        
        for cluster, df in sorted(cluster_matrices.items(), key=lambda x: int(x[0])):
            print(f"\nCluster {cluster}:")
            
            # Get prune level columns (exclude 'concept' column)
            prune_cols = [col for col in df.columns if col != 'concept']
            
            # Calculate statistics
            total_concepts = len(df)
            
            for col in prune_cols:
                active_concepts = (df[col] > 0).sum()
                total_strength = df[col].sum()
                avg_strength = df[col][df[col] > 0].mean() if active_concepts > 0 else 0
                max_strength = df[col].max()
                
                print(f"  {col:>8} - Active: {active_concepts:>4}/{total_concepts} ({active_concepts/total_concepts*100:>5.1f}%) | "
                      f"Total Strength: {total_strength:>6.0f} | Avg: {avg_strength:>5.2f} | Max: {max_strength:>4.0f}")
    
    def analyze_concept_degradation(self, model_name, cluster_matrices):
        """Analyze which concepts degrade fastest (lose neurons quickly)"""
        
        print(f"\n{'='*80}")
        print(f"CONCEPT DEGRADATION ANALYSIS: {model_name}")
        print(f"{'='*80}\n")
        
        for cluster, df in sorted(cluster_matrices.items(), key=lambda x: int(x[0])):
            print(f"\nCluster {cluster}:")
            
            prune_cols = [col for col in df.columns if col != 'concept']
            
            if len(prune_cols) < 2:
                print("  Not enough pruning levels for degradation analysis")
                continue
            
            # Calculate degradation rate (% loss from baseline to final)
            baseline_col = prune_cols[0]
            final_col = prune_cols[-1]
            
            # Only look at concepts that existed in baseline
            baseline_concepts = df[df[baseline_col] > 0].copy()
            
            if len(baseline_concepts) == 0:
                continue
            
            baseline_concepts['degradation_rate'] = (
                (baseline_concepts[baseline_col] - baseline_concepts[final_col]) / 
                baseline_concepts[baseline_col] * 100
            )
            
            # Sort by degradation rate
            baseline_concepts = baseline_concepts.sort_values('degradation_rate', ascending=False)
            
            # Most degraded concepts
            print(f"\n  Top 10 Most Degraded Concepts ({baseline_col} → {final_col}):")
            for idx, row in baseline_concepts.head(10).iterrows():
                print(f"    {row['concept']:40} | {row[baseline_col]:>3.0f} → {row[final_col]:>3.0f} neurons ({row['degradation_rate']:>6.1f}% loss)")
            
            # Most stable concepts (among those with high initial strength)
            high_strength = baseline_concepts[baseline_concepts[baseline_col] >= 5].copy()
            if len(high_strength) > 0:
                most_stable = high_strength.nsmallest(10, 'degradation_rate')
                print(f"\n  Top 10 Most Stable High-Strength Concepts:")
                for idx, row in most_stable.iterrows():
                    print(f"    {row['concept']:40} | {row[baseline_col]:>3.0f} → {row[final_col]:>3.0f} neurons ({row['degradation_rate']:>6.1f}% loss)")
    
    def run_full_analysis(self, output_dir='concept_strength_output'):
        """Run complete analysis for all models"""
        
        print(f"{'='*80}")
        print(f"CONCEPT STRENGTH ANALYSIS")
        print(f"{'='*80}")
        
        all_results = {}
        
        for model_name in self.models:
            print(f"\n\n{'='*80}")
            print(f"PROCESSING MODEL: {model_name}")
            print(f"{'='*80}")
            
            # Load data
            prune_data = self.load_model_data(model_name)
            
            if not prune_data:
                print(f"  Skipping {model_name} - no data found")
                continue
            
            # Build matrices
            cluster_matrices = self.build_concept_strength_matrix(model_name, prune_data)
            
            # Save to CSV
            output_path = self.save_cluster_matrices(model_name, cluster_matrices, output_dir)
            
            # Generate statistics
            self.generate_summary_statistics(model_name, cluster_matrices)
            
            # Analyze degradation
            self.analyze_concept_degradation(model_name, cluster_matrices)
            
            all_results[model_name] = {
                'output_path': output_path,
                'cluster_matrices': cluster_matrices
            }
        
        print(f"\n\n{'='*80}")
        print(f"ANALYSIS COMPLETE")
        print(f"{'='*80}")
        print(f"\nOutput directory: {Path(output_dir).absolute()}")
        print(f"\nGenerated files:")
        for model_name, results in all_results.items():
            print(f"\n{model_name}:")
            for cluster in sorted(results['cluster_matrices'].keys(), key=int):
                print(f"  - Cluster{cluster}_conceptstrength.csv")
        
        return all_results


# Example usage
if __name__ == "__main__":
    # Set your base path - should contain BERT/, BOWMAN/, LLAMA/ subdirectories
    base_path = "/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25/Expls"  # Adjust this path
    
    tracker = ConceptStrengthTracker(base_path)
    results = tracker.run_full_analysis(output_dir='concept_strength_output')
    
    print("\n\nDone!")


"""
EXAMPLE OUTPUT:
================================================================================

================================================================================
CONCEPT STRENGTH ANALYSIS
================================================================================


================================================================================
PROCESSING MODEL: BERT
================================================================================

Loading BERT data...
  Loaded: 0% prune, Cluster 1 - 95 unique concepts, 612 units
  Loaded: 0% prune, Cluster 2 - 142 unique concepts, 1843 units
  Loaded: 0% prune, Cluster 3 - 178 unique concepts, 2156 units
  Loaded: 25% prune, Cluster 1 - 88 unique concepts, 459 units
  Loaded: 25% prune, Cluster 2 - 137 unique concepts, 1382 units
  Loaded: 25% prune, Cluster 3 - 174 unique concepts, 1617 units
  Total pruning levels loaded: 6

Building concept strength matrices for BERT...
  Cluster 1: 95 unique concepts across all pruning levels
  Cluster 2: 142 unique concepts across all pruning levels
  Cluster 3: 178 unique concepts across all pruning levels

Saving BERT concept strength matrices to concept_strength_output/BERT...
  Saved: Cluster1_conceptstrength.csv (95 concepts)
  Saved: Cluster2_conceptstrength.csv (142 concepts)
  Saved: Cluster3_conceptstrength.csv (178 concepts)

================================================================================
CONCEPT STRENGTH SUMMARY: BERT
================================================================================

Cluster 1:
       0% - Active:   95/ 95 (100.0%) | Total Strength:    612 | Avg:  6.44 | Max:   28
      25% - Active:   88/ 95 ( 92.6%) | Total Strength:    459 | Avg:  5.22 | Max:   21
      43% - Active:   76/ 95 ( 80.0%) | Total Strength:    349 | Avg:  4.59 | Max:   17
      57% - Active:   63/ 95 ( 66.3%) | Total Strength:    262 | Avg:  4.16 | Max:   14
      68% - Active:   52/ 95 ( 54.7%) | Total Strength:    196 | Avg:  3.77 | Max:   11
      76% - Active:   42/ 95 ( 44.2%) | Total Strength:    147 | Avg:  3.50 | Max:    9

Cluster 2:
       0% - Active:  142/142 (100.0%) | Total Strength:   1843 | Avg: 12.98 | Max:   67
      25% - Active:  137/142 ( 96.5%) | Total Strength:   1382 | Avg: 10.09 | Max:   52
      43% - Active:  118/142 ( 83.1%) | Total Strength:   1050 | Avg:  8.90 | Max:   41
      57% - Active:   96/142 ( 67.6%) | Total Strength:    788 | Avg:  8.21 | Max:   34
      68% - Active:   88/142 ( 62.0%) | Total Strength:    591 | Avg:  6.72 | Max:   28
      76% - Active:   81/142 ( 57.0%) | Total Strength:    442 | Avg:  5.46 | Max:   22

Cluster 3:
       0% - Active:  178/178 (100.0%) | Total Strength:   2156 | Avg: 12.11 | Max:   89
      25% - Active:  174/178 ( 97.8%) | Total Strength:   1617 | Avg:  9.29 | Max:   68
      43% - Active:  154/178 ( 86.5%) | Total Strength:   1229 | Avg:  7.98 | Max:   54
      57% - Active:  128/178 ( 71.9%) | Total Strength:    922 | Avg:  7.20 | Max:   43
      68% - Active:  111/178 ( 62.4%) | Total Strength:    691 | Avg:  6.23 | Max:   36
      76% - Active:   98/178 ( 55.1%) | Total Strength:    518 | Avg:  5.29 | Max:   29

================================================================================
CONCEPT DEGRADATION ANALYSIS: BERT
================================================================================

Cluster 1:

  Top 10 Most Degraded Concepts (0% → 76%):
    [in]                                     |  28 →   0 neurons ( 100.0% loss)
    [on]                                     |  24 →   0 neurons ( 100.0% loss)
    [the]                                    |  22 →   0 neurons ( 100.0% loss)
    [with]                                   |  19 →   0 neurons ( 100.0% loss)
    a                                        |  15 →   0 neurons ( 100.0% loss)
    [at]                                     |  14 →   1 neurons (  92.9% loss)
    [for]                                    |  13 →   1 neurons (  92.3% loss)
    the dog                                  |  12 →   2 neurons (  83.3% loss)
    red                                      |  11 →   2 neurons (  81.8% loss)
    small                                    |  10 →   2 neurons (  80.0% loss)

  Top 10 Most Stable High-Strength Concepts:
    person                                   |  15 →  12 neurons (  20.0% loss)
    animal                                   |  14 →  11 neurons (  21.4% loss)
    water                                    |  13 →  10 neurons (  23.1% loss)
    house                                    |  12 →   9 neurons (  25.0% loss)
    food                                     |  11 →   8 neurons (  27.3% loss)
    city                                     |  10 →   7 neurons (  30.0% loss)
    building                                 |   9 →   6 neurons (  33.3% loss)
    move                                     |   8 →   5 neurons (  37.5% loss)
    walk                                     |   7 →   4 neurons (  42.9% loss)
    run                                      |   7 →   4 neurons (  42.9% loss)

[... Similar output for BOWMAN and LLAMA models ...]

================================================================================
ANALYSIS COMPLETE
================================================================================

Output directory: /path/to/concept_strength_output

Generated files:

BERT:
  - Cluster1_conceptstrength.csv
  - Cluster2_conceptstrength.csv
  - Cluster3_conceptstrength.csv

BOWMAN:
  - Cluster1_conceptstrength.csv
  - Cluster2_conceptstrength.csv
  - Cluster3_conceptstrength.csv

LLAMA:
  - Cluster1_conceptstrength.csv
  - Cluster2_conceptstrength.csv
  - Cluster3_conceptstrength.csv
"""

CONCEPT STRENGTH ANALYSIS


PROCESSING MODEL: BERT

Loading BERT data...
  Loaded: 0.0% prune, Cluster 1 - 519 unique concepts, 1024 units
  Loaded: 0.0% prune, Cluster 2 - 352 unique concepts, 1024 units
  Loaded: 0.0% prune, Cluster 3 - 560 unique concepts, 757 units
  Loaded: 25.0% prune, Cluster 1 - 518 unique concepts, 1024 units
  Loaded: 25.0% prune, Cluster 2 - 366 unique concepts, 1024 units
  Loaded: 25.0% prune, Cluster 3 - 583 unique concepts, 731 units
  Loaded: 43.75% prune, Cluster 1 - 554 unique concepts, 1024 units
  Loaded: 43.75% prune, Cluster 2 - 377 unique concepts, 1024 units
  Loaded: 43.75% prune, Cluster 3 - 588 unique concepts, 768 units
  Loaded: 57.812% prune, Cluster 1 - 520 unique concepts, 1024 units
  Loaded: 57.812% prune, Cluster 2 - 384 unique concepts, 1024 units
  Loaded: 57.812% prune, Cluster 3 - 597 unique concepts, 822 units
  Loaded: 68.359% prune, Cluster 1 - 555 unique concepts, 1024 units
  Loaded: 68.359% prune, Cluster 2 - 379 unique conc

'\nEXAMPLE OUTPUT:\n================================================================================\n\n================================================================================\nCONCEPT STRENGTH ANALYSIS\n================================================================================\n\n\n================================================================================\nPROCESSING MODEL: BERT\n================================================================================\n\nLoading BERT data...\n  Loaded: 0% prune, Cluster 1 - 95 unique concepts, 612 units\n  Loaded: 0% prune, Cluster 2 - 142 unique concepts, 1843 units\n  Loaded: 0% prune, Cluster 3 - 178 unique concepts, 2156 units\n  Loaded: 25% prune, Cluster 1 - 88 unique concepts, 459 units\n  Loaded: 25% prune, Cluster 2 - 137 unique concepts, 1382 units\n  Loaded: 25% prune, Cluster 3 - 174 unique concepts, 1617 units\n  Total pruning levels loaded: 6\n\nBuilding concept strength matrices for BERT...\n  Cluster 1: 95 